# Data Engineer (итерация 1)

# Data Engineer Report

## Контекст
- Источник: `data/raw/fake_job_postings.csv`
- Целевая колонка: `fraudulent`
- Выходной файл: `/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv`
- Бизнес-задача: бинарная классификация мошеннических вакансий для HR-площадки.
- Приоритет метрики: `F1` и `recall` класса 1 при контроле `precision`.

## План очистки
1. Загрузить датасет и проверить размер, типы и пропуски.
2. Выделить числовые, категориальные и текстовые признаки.
3. Удалить признаки с долей пропусков > 70%.
4. Для числовых признаков применить imputing (`mean`/`median`) и clipping по 1/99 перцентилям.
5. Для категориальных признаков применить `mode`-imputation и кодирование по cardinality.
6. Для текстовых признаков сохранить содержимое как есть, но пропуски заполнить пустой строкой, чтобы итоговый cleaned dataset не содержал NaN.
7. Сохранить очищенный датасет и вывести итоговый контроль качества.

In [ ]:
import pandas as pd
import numpy as np

DF = pd.read_csv("data/raw/fake_job_postings.csv")
print(DF.shape)
print(DF.isna().sum())

(17880, 18)
job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64


## Профиль данных

Ниже выполняется профилирование: типы данных, доля пропусков, распределение target и разбиение признаков на числовые, категориальные и текстовые.

In [ ]:
import pandas as pd
import numpy as np

target_col = "fraudulent"

text_columns_candidates = [
    "title", "description", "requirements", "benefits", "company_profile",
    "salary_range", "job_id"
]
forced_categorical = ["employment_type", "required_experience", "required_education", "industry", "function", "department"]

na_share = (DF.isna().mean().sort_values(ascending=False) * 100).round(2)
print("Dtypes:\n", DF.dtypes)
print("\nNaN share (%):\n", na_share)
print("\nTarget distribution:")
print(DF[target_col].value_counts(dropna=False))
print((DF[target_col].value_counts(normalize=True, dropna=False) * 100).round(2))

numeric_cols = [c for c in DF.select_dtypes(include=[np.number]).columns if c != target_col]
object_cols = DF.select_dtypes(include=["object"]).columns.tolist()

text_cols = [c for c in text_columns_candidates if c in DF.columns and c in object_cols and c not in forced_categorical]
categorical_cols = [c for c in forced_categorical if c in DF.columns and c != target_col]
categorical_cols += [
    c for c in object_cols
    if c not in text_cols and c not in categorical_cols and c != target_col
]

profile_df = pd.DataFrame({
    "column": DF.columns,
    "dtype": [str(DF[c].dtype) for c in DF.columns],
    "nan_share_pct": [round(DF[c].isna().mean() * 100, 2) for c in DF.columns],
    "nunique": [DF[c].nunique(dropna=True) for c in DF.columns],
    "group": [
        "target" if c == target_col else
        "numeric" if c in numeric_cols else
        "text" if c in text_cols else
        "categorical" if c in categorical_cols else
        "other"
        for c in DF.columns
    ]
})

print("\nNumeric columns:\n", numeric_cols)
print("\nCategorical columns:\n", categorical_cols)
print("\nText columns:\n", text_cols)
print("\nProfile table:\n", profile_df.sort_values(["group", "nan_share_pct"], ascending=[True, False]).to_string(index=False))

Dtypes:
 job_id                 int64
title                    str
location                 str
department               str
salary_range             str
company_profile          str
description              str
requirements             str
benefits                 str
telecommuting          int64
has_company_logo       int64
has_questions          int64
employment_type          str
required_experience      str
required_education       str
industry                 str
function                 str
fraudulent             int64
dtype: object

NaN share (%):
 salary_range           83.96
department             64.58
required_education     45.33
benefits               40.34
required_experience    39.43
function               36.10
industry               27.42
employment_type        19.41
company_profile        18.50
requirements           15.08
location                1.94
description             0.01
job_id                  0.00
telecommuting           0.00
has_questions           0.00
has

## Стратегия и применение

В этой секции выполняется реальная очистка с учётом QC-фидбека:
- колонки с >70% NaN удаляются;
- `employment_type`, `function`, `industry`, `department` обрабатываются как категориальные;
- текстовые поля не кодируются, но их пропуски заполняются пустой строкой, чтобы в cleaned dataset не осталось NaN;
- числовые признаки заполняются mean/median в зависимости от устойчивости к выбросам и затем clip по 1/99 перцентилям;
- категориальные признаки заполняются mode, далее `one-hot` при `<20` уникальных значениях и `frequency encoding` при `>50` уникальных;
- target не изменяется и не кодируется.

In [ ]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype

DF = pd.read_csv("data/raw/fake_job_postings.csv")

target_col = "fraudulent"
text_columns_candidates = [
    "title", "description", "requirements", "benefits", "company_profile", "salary_range", "job_id"
]
forced_categorical = ["employment_type", "required_experience", "required_education", "industry", "function", "department"]

original_columns = DF.columns.tolist()
actions = []

# 1. Drop columns with >70% NaN except target
nan_ratio = DF.isna().mean()
drop_cols = [c for c in DF.columns if c != target_col and nan_ratio[c] > 0.70]
for c in drop_cols:
    actions.append({"column": c, "strategy": "drop_column", "reason": f">70% NaN ({nan_ratio[c]:.2%})"})
DF = DF.drop(columns=drop_cols)

# 2. Rebuild column groups after dropping
object_cols = DF.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = [c for c in DF.select_dtypes(include=[np.number]).columns if c != target_col]
text_cols = [c for c in text_columns_candidates if c in DF.columns and c in object_cols and c not in forced_categorical]
categorical_cols = [c for c in forced_categorical if c in DF.columns and c != target_col]
categorical_cols += [
    c for c in object_cols
    if c not in text_cols and c not in categorical_cols and c != target_col
]

# 3. Numeric cleaning: clip + impute mean/median
for c in numeric_cols:
    non_na = DF[c].dropna()
    if non_na.empty:
        fill_value = 0
        DF[c] = DF[c].fillna(fill_value)
        actions.append({"column": c, "strategy": "impute_median", "reason": "numeric column had all values missing after load; fallback fill to 0 to remove NaN"})
        continue

    q1 = non_na.quantile(0.25)
    q3 = non_na.quantile(0.75)
    iqr = q3 - q1
    p01 = non_na.quantile(0.01)
    p99 = non_na.quantile(0.99)
    DF[c] = DF[c].clip(lower=p01, upper=p99)
    actions.append({"column": c, "strategy": "clip_1_99", "reason": f"outlier control using p01={p01:.4f}, p99={p99:.4f}"})

    if iqr == 0:
        fill_value = DF[c].median()
        DF[c] = DF[c].fillna(fill_value)
        actions.append({"column": c, "strategy": "impute_median", "reason": "IQR=0; robust fill chosen"})
    else:
        outlier_rate = ((non_na < (q1 - 1.5 * iqr)) | (non_na > (q3 + 1.5 * iqr))).mean()
        skewness = non_na.skew()
        if abs(skewness) < 1 and outlier_rate < 0.05:
            fill_value = DF[c].mean()
            DF[c] = DF[c].fillna(fill_value)
            actions.append({"column": c, "strategy": "impute_mean", "reason": f"approximately symmetric distribution (skew={skewness:.3f}, outlier_rate={outlier_rate:.3f})"})
        else:
            fill_value = DF[c].median()
            DF[c] = DF[c].fillna(fill_value)
            actions.append({"column": c, "strategy": "impute_median", "reason": f"skew/outliers detected (skew={skewness:.3f}, outlier_rate={outlier_rate:.3f})"})

# 4. Text cleaning: keep as is but fill missing with empty string
for c in text_cols:
    missing_before = DF[c].isna().sum()
    DF[c] = DF[c].fillna("")
    actions.append({"column": c, "strategy": "text_as_is + fill_empty_string", "reason": f"text column kept raw for DS featurization; removed {missing_before} NaN"})

# 5. Categorical cleaning: mode + encoding by cardinality
for c in categorical_cols:
    mode_series = DF[c].mode(dropna=True)
    fill_value = mode_series.iloc[0] if not mode_series.empty else "Unknown"
    missing_before = DF[c].isna().sum()
    DF[c] = DF[c].fillna(fill_value)

    nunique = DF[c].nunique(dropna=True)
    if nunique < 20:
        dummies = pd.get_dummies(DF[c], prefix=c, dummy_na=False)
        DF = pd.concat([DF.drop(columns=[c]), dummies], axis=1)
        actions.append({"column": c, "strategy": "mode + one_hot", "reason": f"categorical column, {missing_before} NaN filled with mode='{fill_value}', low cardinality ({nunique})"})
    elif nunique > 50:
        freq_map = DF[c].value_counts(dropna=False) / len(DF)
        DF[c] = DF[c].map(freq_map)
        actions.append({"column": c, "strategy": "mode + frequency_encoding", "reason": f"categorical column, {missing_before} NaN filled with mode='{fill_value}', high cardinality ({nunique})"})
    else:
        actions.append({"column": c, "strategy": "mode_only", "reason": f"categorical column, {missing_before} NaN filled with mode='{fill_value}', medium cardinality ({nunique}) left as single feature"})

# 6. Final safety pass: remove any residual NaN outside target and preserve target untouched
feature_cols = [c for c in DF.columns if c != target_col]
for c in feature_cols:
    if DF[c].isna().sum() > 0:
        if is_numeric_dtype(DF[c]):
            fill_value = DF[c].median() if not DF[c].dropna().empty else 0
            DF[c] = DF[c].fillna(fill_value)
            actions.append({"column": c, "strategy": "final_numeric_fill", "reason": "safety pass to remove residual NaN after transformations"})
        else:
            mode_series = DF[c].mode(dropna=True)
            fill_value = mode_series.iloc[0] if not mode_series.empty else ""
            DF[c] = DF[c].fillna(fill_value)
            actions.append({"column": c, "strategy": "final_categorical_fill", "reason": "safety pass to remove residual NaN after transformations"})

output_path = "/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv"
DF.to_csv(output_path, index=False)

actions_df = pd.DataFrame(actions).drop_duplicates().sort_values(["strategy", "column"]).reset_index(drop=True)
print("Final shape:", DF.shape)
print("Total NaN after cleaning:", DF.isna().sum().sum())
print("Target present:", target_col in DF.columns)
print("Saved to:", output_path)
print("\nActions summary:\n", actions_df.to_string(index=False))

<string>:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
Final shape: (17880, 39)
Total NaN after cleaning: 0
Target present: True
Saved to: /Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv

Actions summary:
              column                       strategy                                                                                                                 reason
   has_company_logo                      clip_1_99                                                                           outlier control using p01=0.0000, p99=1.0000
      has_questi

## Применённые действия

| column | strategy | reason |
|---|---|---|
| Все колонки с >70% NaN | drop_column | Удалены как нерепрезентативные для стабильного EDA/ML pipeline. |
| Числовые признаки | clip_1_99 | Ограничение выбросов по 1 и 99 перцентилям для устойчивости статистик. |
| Числовые признаки с близким к нормальному распределением | impute_mean | Заполнение средним, если распределение симметрично и выбросов мало. |
| Числовые признаки со skew/outliers | impute_median | Робастное заполнение медианой. |
| `employment_type`, `required_experience`, `required_education`, `industry`, `function`, `department` | mode + one-hot / frequency | Явно обработаны как категориальные по cardinality, согласно QC-фидбеку. |
| Категориальные признаки `<20` уникальных | one-hot encoding | Низкая кардинальность, безопасно для размерности. |
| Категориальные признаки `>50` уникальных | frequency encoding | Высокая кардинальность, избегаем взрыва числа столбцов. |
| Текстовые признаки (`title`, `description`, `requirements`, `benefits`, `company_profile` и др.) | as_is + fill_empty_string | Текст сохранён для последующей DS-фичеризации, пропуски устранены для реально очищенного датасета без NaN. |
| Все признаки после трансформаций | final safety fill | Контрольный проход для удаления остаточных NaN перед сохранением. |